# BirdCLEF+ 2026 SED Training (v3)

EfficientNet-B0 + AttentionSEDHead. Direct OGG loading (no cache, no VAD).

| Setting | Value |
|---------|-------|
| Accelerator | GPU T4 x2 |
| Internet | ON |

In [ ]:
import os, time, glob, random, pickle, warnings
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import timm
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from dataclasses import dataclass

warnings.filterwarnings('ignore')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# CONFIG
COMP_DATA = None
for p in ['/kaggle/input/birdclef-2026', '/kaggle/input/competitions/birdclef-2026']:
    if os.path.exists(p): COMP_DATA = p; break
assert COMP_DATA

@dataclass
class Config:
    sr: int = 32000
    chunk_duration: float = 5.0
    n_mels: int = 256
    n_fft: int = 2048
    hop_length: int = 512
    fmin: int = 0
    fmax: int = 16000
    top_db: float = 80.0
    power: float = 2.0
    target_size: tuple = (256, 256)
    backbone: str = 'tf_efficientnet_b0.ns_jft_in1k'
    num_classes: int = 234
    in_channels: int = 3
    dropout: float = 0.1
    drop_path_rate: float = 0.0
    gem_p_init: float = 3.0
    num_epochs: int = 10
    batch_size: int = 32
    lr: float = 1e-3
    lr_min: float = 1e-6
    weight_decay: float = 1e-4
    seed: int = 42
    fp16: bool = True
    n_folds: int = 5
    early_stopping: int = 3
    train_audio: str = f'{COMP_DATA}/train_audio'
    train_csv: str = f'{COMP_DATA}/train.csv'
    output_dir: str = '/kaggle/working'
    @property
    def chunk_samples(self): return int(self.sr * self.chunk_duration)

cfg = Config()
os.makedirs(cfg.output_dir, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(cfg.seed)
print(f'Device: {device}')
print(f'Config: epochs={cfg.num_epochs}, batch={cfg.batch_size}')

In [ ]:
# MODEL
class GEMFreqPool(nn.Module):
    """周波数方向の Generalized Mean Pooling。(B,C,F,T)→(B,C,T)
    ※メルスペクトログラムの前処理ではなく、EfficientNetの後段で
    特徴マップの周波数方向(F)を集約し、時間方向(T)だけ残す
    → SED Headが「いつ鳴いたか」を時間方向で判定できるようにする
    p=1: Average Pooling, p→∞: Max Pooling, p=3(初期値): 大きい値を重視する平均
    pは学習可能パラメータ (nn.Parameter) で、学習中に最適値に更新される"""
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p_init))
        self.eps = eps
    def forward(self, x):
        # clamp: 値の下限制限 (min=1.0: pが1未満だと不安定, min=eps: 0のp乗を防ぐ)
        p = self.p.clamp(min=1.0)
        return x.clamp(min=self.eps).pow(p).mean(dim=2).pow(1.0 / p)
        # mean(dim=2): 周波数方向で平均, pow(1/p): p乗根で元のスケールに戻す

class AttentionSEDHead(nn.Module):
    """時間方向の Attention Pooling + 分類。
    FC: 特徴量変換 (EfficientNet出力の次元変換)
    att_conv: 各時間フレームの重要度 (「いつ鳴いたか」)
    cls_conv: 各時間フレームの分類スコア (「何が鳴いているか」)
    最終予測 = 重要度 × 分類スコア の時間方向合計"""
    def __init__(self, feat_dim, num_classes, dropout=0.1):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(feat_dim, feat_dim), nn.ReLU(inplace=True), nn.Dropout(dropout))
        self.att_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)  # 各時間フレームの重要度 (「いつ鳴いたか」)
        self.cls_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)  # 各時間フレームの分類スコア (「何が鳴いているか」)
    def forward(self, x):
        x = self.fc(x.permute(0, 2, 1)).permute(0, 2, 1)  # 特徴量変換
        # tanh: 値を-1~+1に圧縮 (極端な値を抑え複数フレームに分散注目させる)
        # softmax: 時間方向で合計1.0に正規化 (「いつ鳴いたか」の確率分布)
        att = F.softmax(torch.tanh(self.att_conv(x)), dim=-1)
        cls = self.cls_conv(x)  # 各フレームの分類スコア
        # 重要度 × 分類スコア を時間方向で合計 → クリップ全体の予測
        clipwise = (att * cls).sum(dim=-1)
        return {'clipwise_logit': clipwise, 'framewise_logit': cls.permute(0, 2, 1)}

class SEDModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(
            cfg.backbone, pretrained=True, in_chans=cfg.in_channels,
            features_only=False, global_pool='', num_classes=0,
            drop_path_rate=cfg.drop_path_rate)
        self.gem_pool = GEMFreqPool(p_init=cfg.gem_p_init)
        self.head = AttentionSEDHead(self.backbone.num_features, cfg.num_classes, cfg.dropout)
    def forward(self, x):
        # 音声(B,3,256,256) → EfficientNet(B,C,F,T) → GEMFreqPool(B,C,T) → SED Head(B,234)
        return self.head(self.gem_pool(self.backbone(x)))

print('Model OK')

In [ ]:
# MEL TRANSFORM
class MelSpectrogramTransform(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.mel = T.MelSpectrogram(
            sample_rate=cfg.sr, n_fft=cfg.n_fft, hop_length=cfg.hop_length,
            n_mels=cfg.n_mels, f_min=cfg.fmin, f_max=cfg.fmax, power=cfg.power)
        self.db = T.AmplitudeToDB(stype='power', top_db=cfg.top_db)
        self.resize = torchvision.transforms.Resize(cfg.target_size, antialias=True)
    @torch.no_grad()
    def forward(self, waveforms):
        mel = self.db(self.mel(waveforms))
        mel = self.resize(mel)
        B = mel.shape[0]
        flat = mel.reshape(B, -1)
        mn = flat.min(dim=1, keepdim=True)[0].unsqueeze(-1)
        mx = flat.max(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel = (mel - mn) / (mx - mn + 1e-7)
        return mel.unsqueeze(1).repeat(1, 3, 1, 1)

mel_transform = MelSpectrogramTransform(cfg)
print('Mel transform OK')

In [ ]:
# DATASET - direct OGG loading (no cache, no VAD)
class BirdSEDDataset(Dataset):
    def __init__(self, df, cfg, le, mode='train'):
        self.df = df.reset_index(drop=True)
        self.cfg = cfg
        self.le = le
        self.mode = mode
        self.num_classes = cfg.num_classes
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.cfg.train_audio, row['filename'])
        try:
            waveform, _ = librosa.load(path, sr=self.cfg.sr, mono=True)
        except Exception:
            waveform = np.zeros(self.cfg.chunk_samples, dtype=np.float32)
        target = self.cfg.chunk_samples
        if len(waveform) > target:
            if self.mode == 'train':
                start = random.randint(0, len(waveform) - target)
            else:
                start = 0
            waveform = waveform[start:start + target]
        elif len(waveform) < target:
            waveform = np.pad(waveform, (0, target - len(waveform)))
        label = np.zeros(self.num_classes, dtype=np.float32)
        try:
            label[self.le.transform([str(row['primary_label'])])[0]] = 1.0
        except ValueError:
            pass
        return torch.tensor(waveform, dtype=torch.float32), torch.tensor(label)

print('Dataset OK')

In [ ]:
# TRAINING LOOP
def train_one_epoch(model, loader, criterion, optimizer, scaler, mel_tf, device, cfg, epoch):
    model.train()
    mel_tf = mel_tf.to(device)
    total_loss, n = 0, len(loader)
    t0 = time.time()
    for i, (waveforms, labels) in enumerate(loader):
        waveforms, labels = waveforms.to(device), labels.to(device)
        mel = mel_tf(waveforms)
        optimizer.zero_grad(set_to_none=True)
        if cfg.fp16:
            with autocast():
                out = model(mel)
                loss = criterion(out['clipwise_logit'], labels)
                fw_max = out['framewise_logit'].max(dim=1)[0]
                loss = loss + criterion(fw_max, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            out = model(mel)
            loss = criterion(out['clipwise_logit'], labels)
            fw_max = out['framewise_logit'].max(dim=1)[0]
            loss = loss + criterion(fw_max, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        total_loss += loss.item()
        if (i+1) % 100 == 0 or (i+1) == n:
            elapsed = time.time() - t0
            print(f'  Epoch {epoch:02d} [{i+1}/{n}] loss={loss.item():.4f} elapsed={elapsed/60:.1f}m', flush=True)
    return total_loss / n

@torch.no_grad()
def validate(model, loader, criterion, mel_tf, device, cfg):
    model.eval()
    mel_tf = mel_tf.to(device)
    total_loss = 0
    all_preds, all_labels = [], []
    for waveforms, labels in loader:
        waveforms, labels = waveforms.to(device), labels.to(device)
        mel = mel_tf(waveforms)
        if cfg.fp16:
            with autocast():
                out = model(mel)
                loss = criterion(out['clipwise_logit'], labels)
        else:
            out = model(mel)
            loss = criterion(out['clipwise_logit'], labels)
        total_loss += loss.item()
        all_preds.append(torch.sigmoid(out['clipwise_logit']).cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    y_true = np.vstack(all_labels)
    y_pred = np.vstack(all_preds)
    aps = []
    for c in range(y_true.shape[1]):
        idx = np.argsort(y_pred[:, c])[::-1]
        ts = y_true[:, c][idx]
        padded = np.concatenate([np.ones(5, dtype=np.float32), ts])
        tp_cum = np.cumsum(padded)
        prec = tp_cum / (np.arange(len(padded)) + 1)
        aps.append(np.sum(prec * padded) / (tp_cum[-1] + 1e-10))
    return total_loss / len(loader), float(np.mean(aps))

print('Training loop OK')

In [ ]:
# RUN TRAINING - 全データ + アップサンプリング
df = pd.read_csv(cfg.train_csv).dropna(subset=['primary_label']).reset_index(drop=True)
print(f'Original samples: {len(df)}')

sub = pd.read_csv(f'{COMP_DATA}/sample_submission.csv')
all_species = sorted([c for c in sub.columns if c != 'row_id'])
le = LabelEncoder()
le.fit(all_species)
print(f'Classes: {len(le.classes_)}')

# アップサンプリング: 少数クラスを中央値まで繰り返し増やす
counts = df['primary_label'].value_counts()
target_count = int(counts.median())
print(f'Class distribution: min={counts.min()}, median={target_count}, max={counts.max()}')

dfs = [df]
upsampled = 0
for label, count in counts.items():
    if count < target_count:
        subset = df[df['primary_label'] == label]
        n_repeat = target_count - count
        repeated = subset.sample(n=n_repeat, replace=True, random_state=cfg.seed)
        dfs.append(repeated)
        upsampled += n_repeat

df_up = pd.concat(dfs, ignore_index=True)
print(f'After upsampling: {len(df_up)} samples (+{upsampled})')
n_upsampled_classes = sum(1 for label in counts.index if counts[label] < target_count)
print(f'  Upsampled classes: {n_upsampled_classes}')

# 全データで学習 (validation なし、10エポック固定)
print(f'Training on ALL data (no validation split)')
train_ds = BirdSEDDataset(df_up, cfg, le, 'train')
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=0, pin_memory=True, drop_last=True)
print(f'Batches: train={len(train_loader)}')

model = SEDModel(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.num_epochs, eta_min=cfg.lr_min)
scaler = GradScaler(enabled=cfg.fp16)
criterion = nn.BCEWithLogitsLoss()

for epoch in range(1, cfg.num_epochs + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler,
                                 mel_transform, device, cfg, epoch)
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch:02d} | train_loss={train_loss:.4f} | lr={lr:.2e}', flush=True)
    torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                'train_loss': train_loss},
               f'{cfg.output_dir}/best_fold0.pt')

with open(f'{cfg.output_dir}/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
print(f'\nTraining complete: {cfg.num_epochs} epochs on {len(df_up)} samples')

In [ ]:
# Verify outputs
for f in sorted(glob.glob(f'{cfg.output_dir}/*')):
    sz = os.path.getsize(f) / 1024
    print(f'  {os.path.basename(f):35s} {sz:.1f} KB')